# SkyRL Training on Google Colab

This notebook runs **SkyRL tx** (local Tinker API server) on Google Colab with GPU support.

## Features
- **Local Tinker API**: Run SkyRL tx server that implements the Tinker API
- **GRPO/PPO Training**: Use any tinker-cookbook recipe
- **vLLM Inference**: High-throughput inference with vLLM
- **Weights & Biases**: Experiment tracking

## References
- [SkyRL Documentation](https://docs.skyrl.ai/docs)
- [Tinker API Docs](https://tinker-docs.thinkingmachines.ai)
- [Tinker Cookbook](https://github.com/thinkingmachines/tinker-cookbook)

## 1. Setup Environment

In [ ]:
# Mount Google Drive (optional - for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install dependencies
!pip install -q uv
!pip install -q wandb datasets math-verify latex2sympy2-extended trl peft accelerate

# Upgrade pip
!pip install --upgrade pip

## 2. Configure API Keys

In [ ]:
# @title API Keys

WANDB_API_KEY = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN

# Login to W&B
!wandb login "$WANDB_API_KEY"

## 3. Configure Training

In [ ]:
# @title Training Configuration

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # @param {type:"string"}
LORA_RANK = 32  # @param {type:"integer"}
EPOCHS = 20  # @param {type:"integer"}
LEARNING_RATE = 1e-6  # @param {type:"number"}
ALGORITHM = "grpo"  # @param {type:"string", options:["grpo", "ppo", "reinforce"]}
ENVIRONMENT = "gsm8k"  # @param {type:"string", options:["gsm8k", "math"]}

TINKER_PORT = 8000  # @param {type:"integer"}
VLLM_PORT = 7999  # @param {type:"integer"}

## 4. Clone and Setup SkyRL

In [ ]:
%%bash

cd /content

# Clone SkyRL
git clone --depth 1 --branch skyrl_train-v0.4.0 \
    https://github.com/NovaSky-AI/SkyRL.git SkyRL

cd SkyRL/skyrl-train

# Create venv
uv venv --python 3.12 --seed

# Install with vLLM support
source .venv/bin/activate
uv sync --extra vllm --extra gpu --extra tinker

echo "=== SkyRL Setup Complete ==="

## 5. Start SkyRL tx Server (Tinker API)

In [ ]:
%%bash --bg --proc server_proc

cd /content/SkyRL/skyrl-train
source .venv/bin/activate

# Start SkyRL tx Tinker API server
uv run --extra gpu --extra tinker -m skyrl.tinker.api \
    --base-model "$MODEL_NAME" \
    --port $TINKER_PORT \
    --backend-config '{"tensor_parallel_size": 1, "max_lora_adapters": 3}' \
    > /content/skyrl_server.log 2>&1 &

echo "SkyRL tx server starting..."
sleep 15
cat /content/skyrl_server.log

In [ ]:
# Check if server is running
import time
import requests

time.sleep(5)

try:
    response = requests.get(f'http://localhost:{TINKER_PORT}/health', timeout=5)
    print(f'Server status: {response.status_code}')
    print(response.json())
except Exception as e:
    print(f'Server not ready yet: {e}')
    print('Check /content/skyrl_server.log for details')

## 6. Run Training with Tinker Cookbook

In [ ]:
%%bash

cd /content/SkyRL/skyrl-train
source .venv/bin/activate

# Set environment
export TINKER_API_KEY="tml-dummy"
export TINKER_BASE_URL="http://localhost:$TINKER_PORT"
export WANDB_API_KEY="$WANDB_API_KEY"
export HF_TOKEN="$HF_TOKEN"

# Prepare GSM8K data
mkdir -p /content/data/gsm8k
uv run -- python examples/gsm8k/gsm8k_dataset.py --output_dir /content/data/gsm8k

# Run GRPO training
uv run --isolated --extra vllm -m skyrl_train.entrypoints.main_base \
    data.train_data="['/content/data/gsm8k/train.parquet']" \
    data.val_data="['/content/data/gsm8k/validation.parquet']" \
    trainer.algorithm.advantage_estimator="grpo" \
    trainer.policy.model.path="$MODEL_NAME" \
    trainer.placement.colocate_all=false \
    trainer.strategy=fsdp2 \
    trainer.placement.policy_num_gpus_per_node=1 \
    trainer.placement.ref_num_gpus_per_node=1 \
    trainer.placement.policy_num_nodes=1 \
    trainer.placement.ref_num_nodes=1 \
    generator.num_inference_engines=1 \
    generator.inference_engine_tensor_parallel_size=1 \
    trainer.epochs=$EPOCHS \
    trainer.eval_batch_size=256 \
    trainer.eval_before_train=true \
    trainer.eval_interval=5 \
    trainer.train_batch_size=256 \
    trainer.policy_mini_batch_size=64 \
    trainer.micro_forward_batch_size_per_gpu=40 \
    trainer.micro_train_batch_size_per_gpu=40 \
    trainer.ckpt_interval=10 \
    trainer.max_prompt_length=512 \
    generator.sampling_params.max_generate_length=1024 \
    trainer.policy.optimizer_config.lr=$LEARNING_RATE \
    trainer.algorithm.use_kl_loss=true \
    generator.backend=vllm \
    generator.run_engines_locally=true \
    generator.weight_sync_backend=local \
    generator.async_engine=true \
    generator.batched=true \
    environment.env_class=$ENVIRONMENT \
    generator.n_samples_per_prompt=5 \
    generator.gpu_memory_utilization=0.8 \
    trainer.logger=wandb \
    trainer.project_name="skyrl-colab" \
    trainer.run_name="colab-$MODEL_NAME-$ALGORITHM" \
    trainer.resume_mode=null \
    trainer.ckpt_path="/content/drive/MyDrive/skyrl_checkpoints"

## 7. Alternative: Use Tinker Cookbook Scripts

In [ ]:
# After the server is running, you can also use any tinker-cookbook recipe:

%%bash

cd /content/SkyRL/skyrl-train
source .venv/bin/activate

# Install tinker-cookbook
uv pip install tinker-cookbook

# Set environment
export TINKER_API_KEY="tml-dummy"
export TINKER_BASE_URL="http://localhost:$TINKER_PORT"
export WANDB_API_KEY="$WANDB_API_KEY"

# Run a tinker-cookbook recipe
# python -m tinker_cookbook.recipes.math_rl.train \
#     base_url=$TINKER_BASE_URL \
#     model_name=$MODEL_NAME \
#     lora_rank=$LORA_RANK \
#     wandb_project=skyrl-colab

## 8. Cleanup

In [ ]:
# Stop the server
!pkill -f 'skyrl.tinker.api'

print("SkyRL tx server stopped")